In [1]:
import numpy as np
import tifffile
import os
from pathlib import Path
import re
from natsort import natsorted
import zarr
from enum import Enum

In [2]:
green_channel = "Z:\\shrofflab\\CND-1_RedUntwisting_A\\Lineaging\\041423\\CND-1_BV514_Overnight\\BV514_CND1\\Pos1\\SPIMB\\Reg_Sample\\For_Lineaging\\StarryNite\\tif\\"

In [3]:
red_channel = "Z:\\shrofflab\\CND-1_RedUntwisting_A\\Lineaging\\041423\\CND-1_BV514_Overnight\\BV514_CND1\\Pos1\\SPIMB\\Reg_Sample\\For_Lineaging\\StarryNite\\tifr\\"

In [4]:
save_dir = "Z:\\shrofflab\\CND-1_RedUntwisting_A\\Lineaging\\041423\\CND-1_BV514_Overnight\\BV514_CND1\\Pos1\\SPIMB\\Reg_Sample\\For_Lineaging\\StarryNite\\"

In [5]:
class Channel(Enum):
    GREEN = 'green'
    RED = 'red'

In [6]:
def convert_lineage_data_to_zarr(source_dir, save_dir: directory, channel: str, w=False, **kwargs):

    '''
    Reformats all tif slice data for a lineaging time series to zarr. Does not save to directory by default, set 'w=True' to keep files.)

    convert_lineage_data_to_zarr(source_dir, channel: str, save_path: directory)
    source_dir = <PATH>
    save_dir = <PATH>
    channel = ['green', 'red']
    w = False

    Keyword arguments:
        file_name = str
    '''

    nm = Channel(channel)
    source_dir = Path(source_dir)
    save_dir = Path(save_dir)

    if save_dir == source_dir:
        raise Exception('Cannot save to source_dir, please input a new directory.')

    kwargs = kwargs
    if len(kwargs) != 0 and 'file_name' in kwargs.keys():
        file_name = kwargs['file_name']
        if file_name.endswith('.zarr'):
            pass
        else:
            file_name = file_name + '.zarr'

    if len(kwargs) != 0 and 'file_name' not in kwargs.keys():
        raise Exception('Invalid keyword arguments, please input file_name=<string>')

    if len(kwargs) == 0 and nm == Channel.GREEN:
        file_name = 'green_channel.zarr'
        
    if len(kwargs) == 0 and nm == Channel.RED:
        file_name = 'red_channel.zarr'

    save_dir = save_dir.joinpath(file_name)
    print(f"Save directory set to:\n{str(save_dir)}")

    if w == False:
        print("\nWriting disabled")
    if w == True:
        print("\nWriting enabled")

    confirmation = None
    while confirmation not in ['y','Y','n','N']:
        confirmation = input("Confirm save directory [y/n]: ")
    
        if confirmation == 'n':
            print('File conversion cancelled')
            return None
    
        if confirmation == 'y':
            print('Save directory confirmed')

        else:
            print('Please type [y] or [n]')

    print(f"\nInitializing tif to zarr conversion:")
    # Build, sort, and extract list of all timepoints and slices in the folder containing the source data
    with os.scandir(str(source_dir)) as entries:
        files = [entry.name for entry in entries if entry.name.endswith('.tif')]
    tifs = natsorted(files)
    t_all = list({re.search(r't\d{3}', x).group() for x in tifs})
    t_all = natsorted(t_all)

    # Load Data as numpy arrays
    img_list = []
    for t in t_all:

        # Build list of all slice file paths in a single timepoint
        file_paths = []
        for tif in tifs:
            sli = re.search(f'{t}',tif)
            if sli != None:
                file_paths.append(str(source_dir)+ '\\' + sli.string)

        # Open tif data for each slice and input to a list which will be used to create the 3D array
        tiff_3D = []
        for file_path in file_paths:
            with tifffile.TiffFile(file_path) as img:
                sli = img.asarray()
                tiff_3D.append(sli)
        print(np.array(tiff_3D).shape)
        # Collect all 3D image data into a list ordered by timepoint
        img_list.append(np.array(tiff_3D))
        print(f"{t} has been read")
        

    # Convert list of 3D arrays into a 4D array, green channel assumed to start at t000
    print(len(img_list))
    t_stack = np.array(img_list)
    if nm == Channel.GREEN:
        shape = t_stack.shape

    # Convert list of 3D arrays into a 4D array and add buffer frames, green channel assumed to start at t201
    if nm == Channel.RED:
        print("\nAdding buffer to red channel t-dimension")
        fill = np.zeros_like(t_stack[0])[np.newaxis,:].repeat(200,axis=0)
        t_stack = np.concatenate((fill,t_stack),axis=0)
        shape = t_stack.shape

    x,y,z = shape[1:]
    print(f'\nSlices converted to zarr array.\nArray dimensions: {shape}')
    print('Saving array...')

    if w == True:
        # Create a persistent 4D Zarr array
        z = zarr.create(
            shape=shape, 
            chunks=(1, x, y, z),
            dtype='f4', 
            store=str(save_dir), 
            overwrite=True
        )
        
        # Assign your 4D NumPy array to the Zarr store
        z[:] = t_stack
        print(f"\nArray saved to {str(save_dir)}")

    return t_stack

In [7]:
t_stack = convert_lineage_data_to_zarr(source_dir=green_channel, save_dir=save_dir, w=False, channel='green')

Save directory set to:
Z:\shrofflab\CND-1_RedUntwisting_A\Lineaging\041423\CND-1_BV514_Overnight\BV514_CND1\Pos1\SPIMB\Reg_Sample\For_Lineaging\StarryNite\red_channel.zarr

Writing enabled


Confirm save directory [y/n]:  y


Save directory confirmed

Initializing tif to zarr conversion:
(201, 515, 501)
t201 has been read
(201, 515, 501)
t202 has been read
(201, 515, 501)
t203 has been read
(201, 515, 501)
t204 has been read
(201, 515, 501)
t205 has been read
(201, 515, 501)
t206 has been read
(201, 515, 501)
t207 has been read
(201, 515, 501)
t208 has been read
(201, 515, 501)
t209 has been read
(201, 515, 501)
t210 has been read
(201, 515, 501)
t211 has been read
(201, 515, 501)
t212 has been read
(201, 515, 501)
t213 has been read
(201, 515, 501)
t214 has been read
(201, 515, 501)
t215 has been read
(201, 515, 501)
t216 has been read
(201, 515, 501)
t217 has been read
(201, 515, 501)
t218 has been read
(201, 515, 501)
t219 has been read
(201, 515, 501)
t220 has been read
(201, 515, 501)
t221 has been read
(201, 515, 501)
t222 has been read
(201, 515, 501)
t223 has been read
(201, 515, 501)
t224 has been read
(201, 515, 501)
t225 has been read
(201, 515, 501)
t226 has been read
(201, 515, 501)
t227 has be